# Superstore Sales Data Cleaning & Preparation

**Author:** Honey Bhatt  
**Date:** March 2026  
**Dataset:** Superstore Retail Sales (9,994 records, 21 columns)  

## Objective
Clean and prepare the raw Superstore dataset for MySQL database ingestion and Tableau dashboard analysis.

## Steps
1. Load and inspect raw data
2. Parse and standardize date columns
3. Rename columns for SQL compatibility
4. Engineer new features (Year, Month, Profit Margin)
5. Validate data quality
6. Export cleaned data and SQL insert file

## Step 1: Load Raw Data

In [5]:
import pandas as pd

In [6]:
# Load raw Superstore dataset
df = pd.read_csv("archive-3/Superstore.csv", encoding='latin1')

In [7]:
# Preview first 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,13-06-2013,17-06-2013,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Step 2: Inspect Data Shape and Types

In [8]:
# Check dataset dimensions: 9,994 rows x 21 columns
df.shape

(9994, 21)

In [9]:
# Check column types and null counts — no missing values found
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

## Step 3: Parse Date Columns

In [10]:
# Convert Order Date from string (dd-mm-yyyy) to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Order Date'].head()

0   2013-11-09
1   2013-11-09
2   2013-06-13
3   2012-10-11
4   2012-10-11
Name: Order Date, dtype: datetime64[ns]

In [11]:
# Convert Ship Date from string (dd-mm-yyyy) to datetime
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')
df['Ship Date'].head()

0   2013-11-12
1   2013-11-12
2   2013-06-17
3   2012-10-18
4   2012-10-18
Name: Ship Date, dtype: datetime64[ns]

## Step 4: Rename Columns for SQL Compatibility

In [12]:
# Strip spaces, replace spaces and hyphens with underscores for MySQL compatibility
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
df.columns

Index(['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode',
       'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State',
       'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category',
       'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

## Step 5: Feature Engineering

In [13]:
# Extract time features from Order Date for time-series analysis
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.month_name()

In [14]:
# Calculate Profit Margin = Profit / Sales
df['Profit_Margin'] = df['Profit'] / df['Sales']
df.Profit_Margin.head()

0    0.1600
1    0.3000
2    0.4700
3   -0.4000
4    0.1125
Name: Profit_Margin, dtype: float64

## Step 6: Data Quality Validation

In [15]:
# Check for invalid sales values (should be 0 rows)
df[df['Sales'] <= 0]

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,Year,Month,Month_Name,Profit_Margin


In [16]:
# Check for invalid discounts > 100% (should be 0 rows)
df[df['Discount'] > 1]

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,Year,Month,Month_Name,Profit_Margin


In [17]:
# Inspect negative profit rows — 1,871 rows have negative profit
# These are valid (high discounts causing losses) and kept for analysis
df[df['Profit'] < 0]

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,Year,Month,Month_Name,Profit_Margin
3,4,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2012,10,October,-0.400000
14,15,US-2012-118983,2012-11-22,2012-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.8100,5,0.80,-123.8580,2012,11,November,-1.800000
15,16,US-2012-118983,2012-11-22,2012-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,Binders,Storex DuraTech Recycled Plastic Frosted Binders,2.5440,3,0.80,-3.8160,2012,11,November,-1.500000
23,24,US-2014-156909,2014-07-17,2014-07-19,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,Philadelphia,...,Chairs,"Global Deluxe Stacking Chair, Gray",71.3720,2,0.30,-1.0196,2014,7,July,-0.014286
27,28,US-2012-150630,2012-09-17,2012-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,7,0.50,-1665.0522,2012,9,September,-0.540000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9920,9921,CA-2013-149272,2013-03-16,2013-03-20,Standard Class,MY-18295,Muhammed Yedwab,Corporate,United States,Bryan,...,Binders,"GBC Pre-Punched Binding Paper, Plastic, White,...",22.3860,7,0.80,-35.8176,2013,3,March,-1.600000
9921,9922,CA-2011-111360,2011-11-24,2011-11-30,Standard Class,AT-10435,Alyssa Tate,Home Office,United States,Akron,...,Binders,Acco Expandable Hanging Binders,5.7420,3,0.70,-4.5936,2011,11,November,-0.800000
9931,9932,CA-2012-104948,2012-11-13,2012-11-17,Standard Class,KH-16510,Keith Herrera,Consumer,United States,San Bernardino,...,Bookcases,O'Sullivan Living Dimensions 3-Shelf Bookcases,683.3320,4,0.15,-40.1960,2012,11,November,-0.058824
9937,9938,CA-2013-164889,2013-06-04,2013-06-07,Second Class,CP-12340,Christine Phan,Corporate,United States,Los Angeles,...,Tables,Hon 61000 Series Interactive Training Tables,71.0880,2,0.20,-1.7772,2013,6,June,-0.025000


In [18]:
# Final preview of cleaned dataset with all new features
df.head()

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,Year,Month,Month_Name,Profit_Margin
0,1,CA-2013-152156,2013-11-09,2013-11-12,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2013,11,November,0.1600
1,2,CA-2013-152156,2013-11-09,2013-11-12,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2013,11,November,0.3000
2,3,CA-2013-138688,2013-06-13,2013-06-17,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2013,6,June,0.4700
3,4,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2012,10,October,-0.4000
4,5,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2012,10,October,0.1125


In [19]:
# Check for duplicate rows — result: 0 duplicates
df.duplicated().sum()

0

## Step 7: Export Cleaned Data

In [20]:
# Export cleaned dataset to CSV
df.to_csv("cleaned_superstore.csv", index=False, encoding="utf-8")

In [21]:
# Generate SQL INSERT statements for MySQL database ingestion
import pandas as pd

df_sql = df.copy()

# Convert dates to string for SQL compatibility
df_sql['Order_Date'] = df_sql['Order_Date'].dt.strftime('%Y-%m-%d')
df_sql['Ship_Date'] = df_sql['Ship_Date'].dt.strftime('%Y-%m-%d')

def sql_value(x):
    if pd.isna(x):
        return "NULL"
    if isinstance(x, str):
        x = x.replace("\\", "\\\\").replace("'", "''")
        return f"'{x}'"
    return str(x)

output_file = "data_insert.sql"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("USE superstore_db;\n\n")
    
    batch_size = 500
    cols = ", ".join(df_sql.columns)
    
    for start in range(0, len(df_sql), batch_size):
        batch = df_sql.iloc[start:start+batch_size]
        values_sql = []
        
        for _, row in batch.iterrows():
            row_values = ", ".join(sql_value(v) for v in row)
            values_sql.append(f"({row_values})")
        
        insert_stmt = (
            f"INSERT INTO cleaned_superstore ({cols}) VALUES\n"
            + ",\n".join(values_sql)
            + ";\n\n"
        )
        f.write(insert_stmt)

print("SQL file created at:", output_file)

SQL file created at: data_insert.sql
